# CS-781 Phase 4 - Court Judgment Prediction (CJPE): all experiments in one notebook

**Setup on Kaggle:** Accelerator *GPU T4 x2*, Internet on, secret `HF` (your Hugging Face token), and these inputs:
`export_gqa`, `export_mla`, `export_retnet` (datasets) and the `Phase1-Arch3` notebook output (contains `export_encdec`).
Run once with `SMOKE = True` in the last cell (~10 min pipeline check), then set `SMOKE = False` and *Save Version -> Save & Run All*.

**Model (shared by every run, so the comparison is about the adaptation technique and the backbone).**
Judgments are long and their end carries the court's reasoning, so each document is reduced to its **last 4 x 512 tokens**.
Every 512-token chunk goes through the Phase-1 backbone (decoders: causal states; enc-dec: bidirectional encoder with a padding mask),
is pooled (masked mean + learned attention), gets a *position-from-the-end* embedding, and the chunk vectors are combined by
attention pooling plus the final-chunk vector -> 2 logits. Training samples 2-4 tail chunks per batch (cheaper, robust to length),
uses a random 10k-document subset per epoch, class-weighted cross-entropy, a 75-minute budget per run and keeps the best dev checkpoint.
Macro-F1 is the metric, so the decision threshold on p(1) is tuned on dev (kept only if a 2-fold check on dev confirms it helps).

| GPU 0 | GPU 1 |
|---|---|
| full fine-tuning - GQA | partial fine-tuning (top 6 layers) - RetNet |
| **LoRMA** (compulsory) - GQA | Houlsby adapters - enc-dec |
| BitFit - MLA | AdaLoRA - RetNet |

then **knowledge distillation** (teacher = mean probability of the six runs on a fixed 8k-document train subset; student = 8-layer GQA,
fully fine-tuned) and a dev-selected **ensemble**. Every run writes `cj_<method>_<arch>/submission.zip`, `results.json` and probabilities;
all zips are collected in `/kaggle/working/submissions/`. All adaptation methods are written from scratch in `cj_peft.py`.

In [1]:
import os
HF_TOKEN = None
for _name in ("HF", "HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret(_name)
        if HF_TOKEN:
            break
    except Exception as e:
        print("secret", _name, "not available:", type(e).__name__)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("No HF token found")
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
%%writefile cj_lib.py
"""CS-781 Phase 4 (CJPE) library: binary judgment prediction on long legal documents with a Phase-1 backbone.
doc -> whitespace-normalised -> last K*CHUNK tokens (the end of a judgment carries the court's reasoning)
-> chunks of CHUNK tokens -> Phase-1 backbone per chunk (+ adaptation method) -> masked mean + attention pooling
-> chunk vectors + position-from-end embedding -> attention pooling over chunks + last-chunk vector -> 2 logits.
The decision threshold on p(label=1) is tuned on dev for macro-F1."""
import os, sys, json, math, time, glob, random, hashlib, types, zipfile, importlib.util, contextlib
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PAD_ID = 1


# ====================================================================== Phase-1 models
def load_module_from_file(path, name):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def file_hash(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()[:10]


def find_p1_export(arch, roots=("/kaggle/input", "/kaggle/working", ".")):
    hits = []
    for r in roots:
        hits += glob.glob(f"{r}/**/export_{arch}/config.json", recursive=True)
    if not hits:                                  # datasets uploaded from an export folder keep files at the top level
        for r in roots:
            for c in sorted(glob.glob(f"{r}/**/config.json", recursive=True)):
                try:
                    if json.load(open(c)).get("arch") == arch and Path(c).with_name("model.pt").exists():
                        hits.append(c)
                except Exception:
                    pass
    return Path(hits[0]).parent if hits else None


def load_p1(export_dir):
    export_dir = Path(export_dir)
    cj = json.load(open(export_dir / "config.json"))
    mod = load_module_from_file(export_dir / "legal_arch.py", f"legal_arch_{cj['arch']}_{file_hash(export_dir / 'legal_arch.py')}")
    bb = mod.build_model(mod.ArchConfig(**cj["cfg"]))
    bb.load_state_dict(torch.load(export_dir / "model.pt", map_location="cpu", weights_only=False))
    return mod, cj["cfg"], bb


def blocks_of(bb):
    return bb.blocks if bb.kind == "decoder" else bb.enc_blocks


def final_norm_of(bb):
    return bb.norm_f if bb.kind == "decoder" else bb.enc_norm


def strip_backbone(bb):
    if bb.kind == "encdec":
        del bb.dec_blocks
        del bb.dec_norm
    bb.lm_head = nn.Identity()
    return bb


def replace_modules(bb, match, factory):
    n = 0
    for blk in blocks_of(bb):
        for name, mod in list(blk.named_modules()):
            if name and match(name, mod):
                parent_name, _, leaf = name.rpartition(".")
                parent = blk.get_submodule(parent_name) if parent_name else blk
                setattr(parent, leaf, factory(mod))
                n += 1
    return n


def replace_linears(bb, targets, factory):
    targets = set(targets)
    return replace_modules(bb, lambda n, m: isinstance(m, nn.Linear) and n.split(".")[-1] in targets, factory)


def make_student_bb(mod, cfg, full_bb, keep):
    cfg = dict(cfg)
    if full_bb.kind == "decoder":
        cfg["n_layers"] = len(keep)
    else:
        cfg["n_enc_layers"] = len(keep)
        cfg["n_dec_layers"] = 1
    st = mod.build_model(mod.ArchConfig(**cfg))
    st.tok_emb.load_state_dict(full_bb.tok_emb.state_dict())
    final_norm_of(st).load_state_dict(final_norm_of(full_bb).state_dict())
    for i, j in enumerate(keep):
        blocks_of(st)[i].load_state_dict(blocks_of(full_bb)[j].state_dict())
    return st, cfg


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


# ---- key-padding mask for the enc-dec encoder (Phase-1 encoder never saw padding)
_ENC_MASK = [None]


def _masked_enc_attn_forward(self, x, cos, sin, cache=None):
    arch = sys.modules[type(self).__module__]
    B, T, _ = x.shape
    q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    k = self.wk(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    v = self.wv(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    q = arch.apply_rope(q, cos[:T], sin[:T])
    k = arch.apply_rope(k, cos[:T], sin[:T])
    m = _ENC_MASK[0]
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=None if m is None else m[:, None, None, :T])
    return self.wo(o.transpose(1, 2).reshape(B, T, -1))


def patch_encoder_padding_mask(bb):
    if bb.kind != "encdec":
        return 0
    for blk in bb.enc_blocks:
        blk.attn.forward = types.MethodType(_masked_enc_attn_forward, blk.attn)
    return len(bb.enc_blocks)


def max_positions(bb):
    rc = getattr(bb, "rope_cos", None)
    return int(rc.shape[0]) if rc is not None else int(getattr(bb.cfg, "max_seq_len", 1024))


def run_backbone(bb, x, valid, train_from=0, ckpt=True):
    """Embeddings -> blocks -> final norm. Blocks below `train_from` run without autograd (frozen bottom),
    trainable blocks use per-block gradient checkpointing."""
    enc = bb.kind == "encdec"
    blocks, norm = (bb.enc_blocks, bb.enc_norm) if enc else (bb.blocks, bb.norm_f)
    cos, sin = bb.rope_cos, bb.rope_sin
    grad = torch.is_grad_enabled()
    if enc:
        _ENC_MASK[0] = valid
    try:
        with torch.set_grad_enabled(grad and bb.tok_emb.weight.requires_grad and train_from == 0):
            h = bb.tok_emb(x)
        for i, blk in enumerate(blocks):
            if grad and i >= train_from:
                h = checkpoint(blk, h, cos, sin, use_reentrant=False) if ckpt else blk(h, cos, sin)
            else:
                with torch.no_grad():
                    h = blk(h, cos, sin)
        return norm(h)
    finally:
        if enc:
            _ENC_MASK[0] = None


# ====================================================================== data
class Doc:
    __slots__ = ("id", "label", "ids", "nwords", "key")


LABEL_KEYS = ("label", "labels", "gold", "y", "outcome", "decision", "target")


def parse_cjpe(path):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        if len(raw) == 1 and isinstance(next(iter(raw.values())), list):
            raw = next(iter(raw.values()))
        else:
            raw = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in raw.items()]
    out = []
    for i, r in enumerate(raw):
        did = r.get("id", r.get("case_id", r.get("name", i)))
        text = r.get("text", r.get("document", r.get("case_text", "")))
        if isinstance(text, list):
            text = " ".join(map(str, text))
        lab = None
        for k in LABEL_KEYS:
            if k in r and r[k] is not None:
                lab = r[k]
                break
        if isinstance(lab, list):
            lab = lab[0] if lab else None
        out.append((str(did), str(text or ""), None if lab is None else int(lab)))
    return out, raw


def tokenize(records, tokenizer, keep_tok, normalize=True):
    """Only the tail is needed: cut a generous character tail first (fast), tokenise it, keep the last keep_tok ids."""
    tail_chars = keep_tok * 12
    docs, texts = [], []
    for did, text, lab in records:
        t = text[-tail_chars:]
        texts.append(" ".join(t.split()) if normalize else t)
        d = Doc()
        d.id, d.label, d.nwords = did, (-1 if lab is None else lab), len(text.split())
        d.key = hashlib.md5(" ".join(text.split())[-3000:].encode("utf-8", "ignore")).hexdigest()
        docs.append(d)
    for i in range(0, len(texts), 2000):
        for d, e in zip(docs[i:i + 2000], tokenizer.encode_batch(texts[i:i + 2000])):
            ids = e.ids[-keep_tok:] if e.ids else [PAD_ID]
            d.ids = np.asarray(ids, dtype=np.int32)
    return docs


def make_batch(docs, k, chunk):
    """Last k*chunk tokens of each doc, split from the END into chunks. Columns are right-aligned so column K-1 is
    always the final chunk; missing (short-document) columns are masked. Returns only the real chunks for the backbone."""
    B = len(docs)
    cmask = np.zeros((B, k), dtype=bool)
    pos = np.zeros((B, k), dtype=np.int64)
    rows = []
    for b, d in enumerate(docs):
        ids = d.ids[-k * chunk:]
        n = max(1, math.ceil(len(ids) / chunk))
        L = len(ids)
        pieces = [ids[max(0, L - (j + 1) * chunk): L - j * chunk] for j in range(n)][::-1]     # earliest first
        for j, p in enumerate(pieces):
            col = k - n + j
            cmask[b, col] = True
            pos[b, col] = n - 1 - j                                                         # 0 = last chunk
            rows.append(p if len(p) else np.asarray([PAD_ID], dtype=np.int32))
    Lm = max(len(r) for r in rows)
    x = np.full((len(rows), Lm), PAD_ID, dtype=np.int64)
    valid = np.zeros((len(rows), Lm), dtype=bool)
    for i, r in enumerate(rows):
        x[i, :len(r)] = r
        valid[i, :len(r)] = True
    return (torch.from_numpy(x), torch.from_numpy(valid), torch.from_numpy(cmask), torch.from_numpy(pos))


# ====================================================================== model
class CJPEModel(nn.Module):
    def __init__(self, bb, hcfg):
        super().__init__()
        d, H, dr = bb.cfg.d_model, hcfg["hid"], hcfg["dropout"]
        self.bb, self.train_from, self.ckpt = bb, int(hcfg.get("train_from", 0)), bool(hcfg.get("ckpt", True))
        self.tok_att = nn.Linear(d, 1)
        self.chunk_proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, H), nn.GELU(), nn.Dropout(dr))
        self.pos = nn.Embedding(hcfg["max_chunks"], H)
        nn.init.normal_(self.pos.weight, std=0.02)
        self.doc_att = nn.Sequential(nn.Linear(H, H), nn.Tanh(), nn.Linear(H, 1))
        self.out = nn.Sequential(nn.LayerNorm(2 * H), nn.Dropout(dr), nn.Linear(2 * H, 2))

    def forward(self, x, valid, cmask, pos):
        B, K = cmask.shape
        h = run_backbone(self.bb, x, valid, self.train_from, self.ckpt)
        m = valid.unsqueeze(-1).float()
        mean = (h.float() * m).sum(1) / m.sum(1).clamp(min=1)
        sc = self.tok_att(h).squeeze(-1).float().masked_fill(~valid, -1e4)
        att = (sc.softmax(1).unsqueeze(-1) * h.float()).sum(1)
        zr = self.chunk_proj(torch.cat([mean, att], -1))
        z = zr.new_zeros(B * K, zr.size(-1))
        z[cmask.view(-1)] = zr
        z = z.view(B, K, -1) + self.pos(pos.clamp(max=self.pos.num_embeddings - 1))
        a = self.doc_att(z).squeeze(-1).float().masked_fill(~cmask, -1e4).softmax(1)
        doc = (a.unsqueeze(-1) * z.float()).sum(1)
        return self.out(torch.cat([doc, z[:, -1].float()], -1)).float()


# ====================================================================== AMP
def get_amp(device):
    if device == "cuda":
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        return (lambda: torch.autocast(device_type="cuda", dtype=dt)), dt
    return (lambda: contextlib.nullcontext()), None


def make_scaler(device, amp_dtype):
    en = device == "cuda" and amp_dtype == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=en)


# ====================================================================== metrics
def macro_f1(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    fs = []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        fs.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(fs))


def prec_rec(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    tp = int(((pred == 1) & (y == 1)).sum())
    return tp / max(1, int((pred == 1).sum())), tp / max(1, int((y == 1).sum()))


def best_threshold(y, p):
    y, p = np.asarray(y), np.asarray(p)
    cands = np.unique(np.concatenate([np.quantile(p, np.linspace(0.02, 0.98, 97)), [0.5]]))
    best = (macro_f1(y, (p >= 0.5).astype(int)), 0.5)
    for t in cands:
        f = macro_f1(y, (p >= t).astype(int))
        if f > best[0] + 1e-9:
            best = (f, float(t))
    return best[1], best[0]


def honest_threshold(y, p, seed=0):
    """2-fold over documents: threshold tuned on one half, scored on the other (tells whether tuning really helps)."""
    y, p = np.asarray(y), np.asarray(p)
    idx = np.random.RandomState(seed).permutation(len(y))
    h = len(idx) // 2
    b, a = [], []
    for tr, te in ((idx[:h], idx[h:]), (idx[h:], idx[:h])):
        t, _ = best_threshold(y[tr], p[tr])
        b.append(macro_f1(y[te], (p[te] >= 0.5).astype(int)))
        a.append(macro_f1(y[te], (p[te] >= t).astype(int)))
    return float(np.mean(b)), float(np.mean(a))


# ====================================================================== inference
@torch.no_grad()
def predict(model, docs, k, chunk, bs, device, amp):
    model.eval()
    order = sorted(range(len(docs)), key=lambda i: len(docs[i].ids))
    p = np.zeros(len(docs), dtype=np.float32)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        x, v, cm, pos = make_batch([docs[j] for j in idx], k, chunk)
        with amp():
            lg = model(x.to(device), v.to(device), cm.to(device), pos.to(device))
        p[idx] = lg.float().softmax(-1)[:, 1].cpu().numpy()
    return p


def evaluate(model, docs, cfg, device, amp):
    p = predict(model, docs, cfg["k_infer"], cfg["chunk"], cfg["eval_bs"], device, amp)
    y = np.array([d.label for d in docs])
    t, ft = best_threshold(y, p)
    return dict(f1_05=macro_f1(y, (p >= 0.5).astype(int)), f1_tuned=ft, thr=t), p


# ====================================================================== training
def train(model, train_docs, dev_docs, cfg, device, amp, amp_dtype, teacher=None, ada=None, log=print):
    """cfg: lr head_lr wd warmup epochs docs_per_epoch bs accum kmin kmax k_infer chunk eval_bs grad_clip
    label_smooth class_w max_min seed evals_per_epoch kd_alpha kd_T"""
    rng = random.Random(cfg["seed"])
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    bbp = [p for n, p in named if n.startswith("bb.")]
    hp = [p for n, p in named if not n.startswith("bb.")]
    groups = ([dict(params=bbp, lr=cfg["lr"], base_lr=cfg["lr"], weight_decay=cfg["wd"])] if bbp else []) + \
             [dict(params=hp, lr=cfg["head_lr"], base_lr=cfg["head_lr"], weight_decay=cfg["wd"])]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-6)
    scaler = make_scaler(device, amp_dtype)
    dpe = min(len(train_docs), cfg["docs_per_epoch"])
    spe = math.ceil(dpe / (cfg["bs"] * cfg["accum"]))
    total = spe * cfg["epochs"]
    warm = max(1, int(cfg["warmup"] * total))
    eval_every = max(1, spe // max(1, cfg.get("evals_per_epoch", 1)))
    cw = torch.tensor(cfg["class_w"], dtype=torch.float32, device=device)
    keep = {n for n, _ in named} | {n for n, _ in model.named_buffers() if "rank_mask" in n}
    tot, tr = count_params(model)
    log(f"{len(train_docs)} train docs ({dpe}/epoch) | {spe} steps/epoch x {cfg['epochs']} = {total} steps | "
        f"params {tot / 1e6:.1f}M, trainable {tr / 1e6:.3f}M ({100 * tr / tot:.2f}%)")
    hist, best, best_state, step, t0, ntok = [], -1.0, None, 0, time.time(), 0
    stop = False
    for ep in range(cfg["epochs"]):
        ids = rng.sample(range(len(train_docs)), dpe)
        model.train()
        run, nrun = 0.0, 0
        for b0 in range(0, len(ids), cfg["bs"]):
            batch = [train_docs[i] for i in ids[b0:b0 + cfg["bs"]]]
            k = rng.randint(cfg["kmin"], cfg["kmax"])
            x, v, cm, pos = make_batch(batch, k, cfg["chunk"])
            y = torch.tensor([d.label for d in batch], device=device)
            with amp():
                lg = model(x.to(device), v.to(device), cm.to(device), pos.to(device))
            lg = lg.float()
            loss = F.cross_entropy(lg, y, weight=cw, label_smoothing=cfg["label_smooth"])
            if teacher is not None:
                T = cfg["kd_T"]
                tp = torch.tensor([teacher[d.id] for d in batch], device=device).clamp(1e-4, 1 - 1e-4)
                tl = torch.stack([torch.log1p(-tp), torch.log(tp)], -1)
                kl = F.kl_div((lg / T).log_softmax(-1), (tl / T).log_softmax(-1), log_target=True, reduction="batchmean") * T * T
                loss = cfg["kd_alpha"] * loss + (1 - cfg["kd_alpha"]) * kl
            if ada is not None:
                loss = loss + ada.reg_loss()
            scaler.scale(loss / cfg["accum"]).backward()
            run += loss.item(); nrun += 1; ntok += int(v.sum())
            if (b0 // cfg["bs"] + 1) % cfg["accum"] == 0 or b0 + cfg["bs"] >= len(ids):
                mult = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))
                for g in opt.param_groups:
                    g["lr"] = g["base_lr"] * mult
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for _, p in named], cfg["grad_clip"])
                if ada is not None:
                    ada.update(step)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                step += 1
                el = (time.time() - t0) / 60
                if step % 25 == 0:
                    log(f"  ep {ep + 1} step {step}/{total} loss {run / max(1, nrun):.4f} | {ntok / max(1e-6, el * 60):,.0f} tok/s | {el:.1f} min")
                stop = el > cfg["max_min"]
                if step % eval_every == 0 or stop or step == total:
                    rep, _ = evaluate(model, dev_docs, cfg, device, amp)
                    rec = dict(epoch=ep + 1, step=step, train_loss=run / max(1, nrun), minutes=el, **rep)
                    if rep["f1_tuned"] > best:
                        best = rep["f1_tuned"]
                        best_state = {kk: vv.detach().cpu().clone() for kk, vv in model.state_dict().items() if kk in keep}
                        rec["best"] = True
                    hist.append(rec)
                    log("== " + " | ".join(f"{kk} {vv:.4f}" if isinstance(vv, float) else f"{kk} {vv}" for kk, vv in rec.items()))
                    model.train()
                if stop:
                    log(f"time budget ({cfg['max_min']} min) reached - stopping")
                    break
        if stop:
            break
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        log(f"restored best checkpoint (dev macro-F1 at tuned threshold {best:.4f})")
    return hist, best


# ====================================================================== outputs
def write_prediction(ids, pred, json_path, zip_path):
    out = [{"id": i, "pred": int(v)} for i, v in zip(ids, pred)]
    json.dump(out, open(json_path, "w"))
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")
    return out


def dump_probs(path, run, docs, p):
    json.dump({"run": run, "docs": {d.id: [round(float(q), 6), int(d.label)] for d, q in zip(docs, p)}}, open(path, "w"))

Writing cj_lib.py


In [3]:
%%writefile cj_peft.py
"""Adaptation methods for Phase 4, written from scratch (no `peft` library). apply(method, bb, cfg) -> info dict.
  full    : every backbone weight trainable
  partial : only the top `n_top` blocks + final norm trainable (blocks below run without autograd)
  bitfit  : bias-only tuning (Ben Zaken et al., 2022): the Phase-1 linears have no bias, so zero-initialised
            biases are added to every linear and trained together with the RMSNorm gains; all weights frozen
  lorma   : (I + (alpha/r) B A) W0 x  - multiplicative low-rank (Bihany, Patel, Modi, 2025)  [compulsory]
  adalora : P diag(lambda) Q with importance-based rank pruning + orthogonality regulariser (Zhang et al., 2023)
  adapter : Houlsby bottleneck adapters after every attention and FFN sub-layer (Houlsby et al., 2019)
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from cj_lib import replace_linears, replace_modules, blocks_of, final_norm_of

DEFAULT_TARGETS = ("wq", "wk", "wv", "wo", "wg", "w_dkv", "w_ukv", "w_gate", "w_up", "w_down")


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.r, self.scale = base, r, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))                 # B = 0 -> identity multiplier at init
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.register_buffer("rank_mask", torch.ones(r))

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.Q) * (self.lam * self.rank_mask), self.P) * self.scale

    def orth_loss(self):
        I = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.t() - I) ** 2).sum()


class AdaLoRAController:
    """Cubic budget schedule; importance = EMA(|w*grad|) x EMA(uncertainty), summed over the triplet (P col, lambda, Q row)."""

    def __init__(self, model, init_r, target_r, total_steps, t_init_frac=0.1, t_final_frac=0.2, delta_t=5,
                 beta1=0.85, beta2=0.85, orth_reg=0.1):
        self.layers = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.total, self.dt = total_steps, delta_t
        self.ti, self.tf = int(t_init_frac * total_steps), int(t_final_frac * total_steps)
        self.b0, self.bT = init_r * len(self.layers), target_r * len(self.layers)
        self.b1, self.b2, self.orth = beta1, beta2, orth_reg
        self.ipt = [None] * len(self.layers)
        self.unc = [None] * len(self.layers)

    def reg_loss(self):
        return self.orth * sum(m.orth_loss() for m in self.layers) / len(self.layers)

    def budget(self, step):
        end = self.total - self.tf
        if step < self.ti:
            return self.b0
        if step >= end:
            return self.bT
        return int(self.bT + (self.b0 - self.bT) * (1 - (step - self.ti) / max(1, end - self.ti)) ** 3)

    @torch.no_grad()
    def update(self, step):
        scores = []
        for i, m in enumerate(self.layers):
            g = lambda p: p.grad if p.grad is not None else torch.zeros_like(p)
            s = torch.stack([(m.lam * g(m.lam)).abs(), (m.P * g(m.P)).abs().mean(0), (m.Q * g(m.Q)).abs().mean(1)]).float()
            s = torch.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
            if self.ipt[i] is None:
                self.ipt[i], self.unc[i] = torch.zeros_like(s), torch.zeros_like(s)
            self.ipt[i] = self.b1 * self.ipt[i] + (1 - self.b1) * s
            self.unc[i] = self.b2 * self.unc[i] + (1 - self.b2) * (s - self.ipt[i]).abs()
            scores.append((self.ipt[i] * self.unc[i]).sum(0))
        end = self.total - self.tf
        if step < self.ti or step > end or ((step - self.ti) % self.dt != 0 and step != end):
            return
        b = self.budget(step)
        allsc = torch.cat(scores)
        if b >= allsc.numel():
            return
        thr = torch.topk(allsc, b).values[-1]
        for m, sc in zip(self.layers, scores):
            m.rank_mask.copy_((sc >= thr).float())

    def ranks(self):
        return [int(m.rank_mask.sum().item()) for m in self.layers]


class Adapter(nn.Module):
    """Wraps a sub-layer (attention or FFN): out = f(x); out + up(gelu(down(out))). Up-projection starts at 0."""

    def __init__(self, inner, d, m):
        super().__init__()
        self.inner = inner
        self.down, self.up = nn.Linear(d, m), nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        h = self.inner(*a, **k)
        return h + self.up(F.gelu(self.down(h)))


def apply(method, bb, cfg):
    """The caller has already set requires_grad=False on every backbone parameter."""
    info = {"method": method}
    nL = len(blocks_of(bb))
    if method in ("full", "kd"):
        for p in bb.parameters():
            p.requires_grad = True
        n = sum(1 for _ in bb.parameters())
    elif method == "partial":
        top = cfg["n_top"]
        for blk in list(blocks_of(bb))[nL - top:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in final_norm_of(bb).parameters():
            p.requires_grad = True
        info["train_from"] = nL - top
        n = top
    elif method == "bitfit":
        n = 0
        for blk in blocks_of(bb):
            for mod in blk.modules():
                if isinstance(mod, nn.Linear):
                    if mod.bias is None:
                        mod.bias = nn.Parameter(torch.zeros(mod.out_features))
                    mod.bias.requires_grad = True
                    n += 1
                elif type(mod).__name__ in ("RMSNorm", "LayerNorm"):
                    for p in mod.parameters():
                        p.requires_grad = True
        for p in final_norm_of(bb).parameters():
            p.requires_grad = True
    elif method == "lorma":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: LoRMALinear(l, cfg["r"], cfg["alpha"]))
    elif method == "adalora":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: AdaLoRALinear(l, cfg["init_r"], cfg["alpha"]))
    elif method == "adapter":
        d, m = bb.cfg.d_model, cfg["bottleneck"]
        n = 0
        for blk in blocks_of(bb):
            blk.attn = Adapter(blk.attn, d, m)
            blk.ffn = Adapter(blk.ffn, d, m)
            n += 2
        for blk in blocks_of(bb):                                  # Houlsby: layer norms are trained as well
            for mod in blk.modules():
                if type(mod).__name__ in ("RMSNorm", "LayerNorm"):
                    for p in mod.parameters():
                        p.requires_grad = True
    else:
        raise ValueError(f"unknown method {method}")
    assert n > 0, f"{method}: nothing to adapt"
    info["n_modules"] = n
    print(f"[{method}] applied ({n} modules)", {k: v for k, v in cfg.items() if k != "targets"})
    return info

Writing cj_peft.py


In [4]:
%%writefile cj_run.py
"""Phase 4 CJPE - one experiment. Usage: python cj_run.py <method> <arch> [tag]
method: full | partial | bitfit | lorma | adalora | adapter | kd      arch: gqa | mla | retnet | encdec
Env: CJ_SMOKE=1 (tiny 1-epoch check), CJ_OVERRIDES='{"EPOCHS": 2}' (override any upper-case setting below)."""
import os, sys, json, math, time, random, glob, importlib
from pathlib import Path
import numpy as np
import torch

METHOD, ARCH = sys.argv[1], sys.argv[2]
TAG = sys.argv[3] if len(sys.argv) > 3 else ""
HF_TOKEN = os.environ.get("HF_TOKEN") or None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SEED = 42

# ---- long-document input: the last K chunks of CHUNK tokens (end of the judgment = the court's reasoning)
CHUNK = 512
KMIN, KMAX = 2, 4          # training: random number of tail chunks per batch (robust to length, cheaper)
K_INFER = 4                # inference / dev: last 4 x 512 = 2048 tokens
KEEP_TOK = 4096            # tokens kept per document after tokenisation

# ---- optimisation
EPOCHS = 3
DOCS_PER_EPOCH = 10000     # random subset of training documents per epoch (different each epoch)
BS, ACCUM, EVAL_BS = 4, 4, 16
HEAD_LR, WD, WARMUP, GRAD_CLIP = 1e-3, 0.01, 0.05, 1.0
LABEL_SMOOTH, CLASS_W_POW = 0.05, 0.5
EVALS_PER_EPOCH = 2
MAX_MIN = 75               # training time budget per run (minutes); the best dev checkpoint is kept
HCFG = dict(hid=512, dropout=0.15, max_chunks=8, ckpt=True)
KD_DOCS = 8000             # fixed subset of training docs on which every run writes probabilities (KD teacher signal)

MCFG_ALL = {
    "full":    dict(lr=2e-5),
    "partial": dict(lr=5e-5, n_top=6),
    "bitfit":  dict(lr=1e-3),
    "lorma":   dict(lr=5e-4, r=16, alpha=16),
    "adalora": dict(lr=1e-3, init_r=12, target_r=6, alpha=12, orth_reg=0.1, delta_t=5),
    "adapter": dict(lr=5e-4, bottleneck=64),
    "kd":      dict(lr=3e-5, keep=None, alpha=0.3, T=2.0),     # student = every other layer, fully fine-tuned
}
for _k, _v in json.loads(os.environ.get("CJ_OVERRIDES", "{}")).items():
    globals()[_k] = _v
SMOKE = os.environ.get("CJ_SMOKE") == "1"
if SMOKE:
    EPOCHS, DOCS_PER_EPOCH, MAX_MIN, KD_DOCS = 1, 48, 5, 40
MCFG = dict(MCFG_ALL[METHOD])

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK = ROOT / f"cj_{METHOD}_{ARCH}{TAG}{'_smoke' if SMOKE else ''}"
WORK.mkdir(parents=True, exist_ok=True)
print("device:", device, "| out:", WORK, "| cfg:", MCFG, flush=True)

import cj_lib, cj_peft
from huggingface_hub import hf_hub_download, HfApi
from tokenizers import Tokenizer
amp, AMP_DTYPE = cj_lib.get_amp(device)

# ------------------------------------------------------------------ data
files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
p4 = [f for f in files if "Phase_4" in f]
print("Phase-4 files:", p4)
def fetch(name):
    m = [f for f in p4 if f.endswith("/" + name)]
    assert m, f"{name} not found in {p4}"
    return hf_hub_download(HF_DATASET_REPO, m[0], repo_type="dataset", token=HF_TOKEN)

recs = {}
for part in ("train", "dev", "test"):
    recs[part], raw = cj_lib.parse_cjpe(fetch(f"{part}.json"))
    r0 = recs[part][0]
    print(f"{part}: {len(recs[part])} docs | raw {type(raw).__name__} | keys {list(raw[0].keys()) if isinstance(raw, list) and isinstance(raw[0], dict) else '-'} "
          f"| example id={r0[0]!r} label={r0[2]} text={r0[1][:100]!r}", flush=True)
# de-duplicate training documents by id (single- and multi-instance data were merged)
seen, tr = set(), []
for r in recs["train"]:
    if r[0] not in seen:
        seen.add(r[0]); tr.append(r)
print(f"train after id de-duplication: {len(tr)} (removed {len(recs['train']) - len(tr)})")
recs["train"] = [r for r in tr if r[2] in (0, 1)]
assert all(r[2] in (0, 1) for r in recs["dev"]), "dev labels must be 0/1"
if SMOKE:
    recs["train"], recs["dev"] = recs["train"][:200], recs["dev"][:64]

export_dir = cj_lib.find_p1_export(ARCH)
assert export_dir is not None, f"Phase-1 export for {ARCH} not found under /kaggle/input"
tokenizer = Tokenizer.from_file(str(export_dir / "tokenizer.json"))
t0 = time.time()
docs = {p: cj_lib.tokenize(recs[p], tokenizer, KEEP_TOK) for p in recs}
print(f"tokenised in {time.time() - t0:.0f}s", flush=True)
train_docs, dev_docs, test_docs = docs["train"], docs["dev"], docs["test"]
for p in ("train", "dev", "test"):
    w = np.array([d.nwords for d in docs[p]]); lab = np.array([d.label for d in docs[p]])
    print(f"{p}: words/doc mean {w.mean():.0f} median {np.median(w):.0f} p90 {np.percentile(w, 90):.0f} | "
          f"label counts {dict(zip(*np.unique(lab, return_counts=True)))}")
tk = {d.key for d in train_docs}
print(f"overlap with train (same text tail): dev {sum(d.key in tk for d in dev_docs)} | test {sum(d.key in tk for d in test_docs)}")
cnt = np.bincount([d.label for d in train_docs], minlength=2).astype(float)
CLASS_W = ((cnt.sum() / (2 * cnt)) ** CLASS_W_POW).tolist()
print("class weights:", CLASS_W)
kd_rng = random.Random(1234)
KD_IDS = set(kd_rng.sample([d.id for d in train_docs], min(KD_DOCS, len(train_docs))))

# ------------------------------------------------------------------ model
arch_mod, bcfg, bb = cj_lib.load_p1(export_dir)
P1_PARAMS = sum(p.numel() for p in bb.parameters())
print(f"Phase-1 {ARCH}: {P1_PARAMS / 1e6:.1f}M params, kind {bb.kind}, positions {cj_lib.max_positions(bb)}")
TEACHER, TEACHER_INFO = None, None
if METHOD == "kd":
    tfiles = sorted(set(glob.glob(str(ROOT / "cj_*/probs_kd.json")) + glob.glob("/kaggle/input/**/probs_kd.json", recursive=True)))
    tfiles = [f for f in tfiles if ("_smoke" in f) == SMOKE and Path(f).parent.resolve() != WORK.resolve()]
    assert tfiles, "KD needs probs_kd.json from earlier runs"
    acc, TEACHER_INFO = {}, []
    for f in tfiles:
        j = json.load(open(f))
        res = Path(f).with_name("results.json")
        TEACHER_INFO.append(dict(run=j["run"], **({k: json.load(open(res)).get(k) for k in ("dev_f1_tuned", "params_total")} if res.exists() else {})))
        for k, (p, _) in j["docs"].items():
            acc.setdefault(k, []).append(p)
    TEACHER = {k: float(np.mean(v)) for k, v in acc.items() if len(v) == len(tfiles)}
    print("teacher = mean of", [t["run"] for t in TEACHER_INFO], "|", len(TEACHER), "docs")
    train_docs = [d for d in train_docs if d.id in TEACHER]
    nL = len(cj_lib.blocks_of(bb))
    keep = MCFG["keep"] or list(range(0, nL, 2))
    bb, bcfg = cj_lib.make_student_bb(arch_mod, bcfg, bb, keep)
    print(f"student keeps layers {keep} of {nL}: {sum(p.numel() for p in bb.parameters()) / 1e6:.1f}M params")

cj_lib.strip_backbone(bb)
cj_lib.patch_encoder_padding_mask(bb)          # must happen before adapters wrap the attention modules
for p in bb.parameters():
    p.requires_grad = False
INFO = cj_peft.apply(METHOD, bb, MCFG)
assert CHUNK <= cj_lib.max_positions(bb)
HCFG["train_from"] = INFO.get("train_from", 0)
model = cj_lib.CJPEModel(bb, HCFG).to(device)
tot, trn = cj_lib.count_params(model)
print(f"model {tot / 1e6:.1f}M params | trainable {trn / 1e6:.3f}M | train_from {HCFG['train_from']}", flush=True)

TCFG = dict(lr=MCFG["lr"], head_lr=HEAD_LR, wd=WD, warmup=WARMUP, epochs=EPOCHS, docs_per_epoch=DOCS_PER_EPOCH, bs=BS,
            accum=ACCUM, kmin=KMIN, kmax=KMAX, k_infer=K_INFER, chunk=CHUNK, eval_bs=EVAL_BS, grad_clip=GRAD_CLIP,
            label_smooth=LABEL_SMOOTH, class_w=CLASS_W, max_min=MAX_MIN, seed=SEED, evals_per_epoch=EVALS_PER_EPOCH,
            kd_alpha=MCFG.get("alpha", 1.0), kd_T=MCFG.get("T", 1.0))
ADA = None
if METHOD == "adalora":
    spe = math.ceil(min(len(train_docs), DOCS_PER_EPOCH) / (BS * ACCUM))
    ADA = cj_peft.AdaLoRAController(model, MCFG["init_r"], MCFG["target_r"], spe * EPOCHS, delta_t=MCFG["delta_t"], orth_reg=MCFG["orth_reg"])
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
hist, best = cj_lib.train(model, train_docs, dev_docs, TCFG, device, amp, AMP_DTYPE, teacher=TEACHER, ada=ADA)
TRAIN_MIN = (time.time() - t0) / 60
PEAK = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0

# ------------------------------------------------------------------ dev, test, outputs
rep, p_dev = cj_lib.evaluate(model, dev_docs, TCFG, device, amp)
y_dev = np.array([d.label for d in dev_docs])
h0, h1 = cj_lib.honest_threshold(y_dev, p_dev, SEED)
THR = rep["thr"] if h1 >= h0 else 0.5
pr, rc = cj_lib.prec_rec(y_dev, (p_dev >= THR).astype(int))
print(f"DEV macro-F1 @0.5 {rep['f1_05']:.4f} | @tuned {rep['f1_tuned']:.4f} (thr {rep['thr']:.3f}) | honest 2-fold {h0:.4f}->{h1:.4f} "
      f"| using thr {THR:.3f} | P {pr:.3f} R {rc:.3f}")
t1 = time.time()
p_test = cj_lib.predict(model, test_docs, K_INFER, CHUNK, EVAL_BS, device, amp)
SPS = len(test_docs) / max(1e-6, time.time() - t1)
cj_lib.write_prediction([d.id for d in test_docs], (p_test >= THR).astype(int), WORK / "prediction.json", WORK / "submission.zip")
print(f"test: {len(test_docs)} docs | predicted 1-rate {(p_test >= THR).mean():.3f} (dev gold 1-rate {y_dev.mean():.3f}) | {SPS:.1f} docs/s")
json.dump([{"id": d.id, "gold": int(d.label)} for d in dev_docs], open(WORK / "dev_truth.json", "w"))
cj_lib.write_prediction([d.id for d in dev_docs], (p_dev >= THR).astype(int), WORK / "dev_prediction.json", WORK / "dev_submission.zip")
cj_lib.dump_probs(WORK / "probs_dev.json", WORK.name, dev_docs, p_dev)
cj_lib.dump_probs(WORK / "probs_test.json", WORK.name, test_docs, p_test)
if METHOD != "kd":
    kd_docs = [d for d in docs["train"] if d.id in KD_IDS]
    cj_lib.dump_probs(WORK / "probs_kd.json", WORK.name, kd_docs, cj_lib.predict(model, kd_docs, K_INFER, CHUNK, EVAL_BS, device, amp))
res = dict(run=WORK.name, method=METHOD, arch=ARCH, mcfg=MCFG, hcfg=HCFG, info=INFO, params_total=tot, params_trainable=trn,
           phase1_params=P1_PARAMS, train_min=TRAIN_MIN, peak_gpu_gb=PEAK, test_docs_per_s=SPS, chunk=CHUNK, k_infer=K_INFER,
           dev_f1_05=rep["f1_05"], dev_f1_tuned=rep["f1_tuned"], thr=THR, honest=[h0, h1], dev_precision=pr, dev_recall=rc,
           history=hist, teacher=TEACHER_INFO, n_train=len(train_docs))
json.dump(res, open(WORK / "results.json", "w"), indent=1, default=float)
print(f"\nSUMMARY {WORK.name}: dev macro-F1 {rep['f1_05']:.4f} @0.5 | {rep['f1_tuned']:.4f} tuned | honest {h0:.4f}->{h1:.4f} | "
      f"params {tot / 1e6:.1f}M trainable {trn / 1e6:.2f}M | train {TRAIN_MIN:.1f} min | peak {PEAK:.1f} GB | {SPS:.1f} docs/s")
if METHOD == "kd":
    tf = [t.get("dev_f1_tuned") for t in TEACHER_INFO if t.get("dev_f1_tuned")]
    tp = [t.get("params_total") for t in TEACHER_INFO if t.get("params_total")]
    if tf:
        print(f"KD: student reaches {100 * rep['f1_tuned'] / max(tf):.1f}% of the best teacher's dev F1 ({max(tf):.4f})"
              + (f" with {100 * tot / max(tp):.0f}% of its parameters" if tp else ""))
print("RUN_DONE", WORK.name, flush=True)

Writing cj_run.py


In [5]:
%%writefile cj_ens.py
"""Phase 4 - ensemble of finished runs (mean of p(label=1)), greedy forward selection on dev, threshold tuned on dev."""
import json, glob, sys, zipfile
from pathlib import Path
import numpy as np
sys.path.insert(0, ".")
from cj_lib import macro_f1, best_threshold, honest_threshold, prec_rec

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
dirs = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/working/cj_*/probs_test.json") + glob.glob("/kaggle/input/**/probs_test.json", recursive=True)
               if "_smoke" not in p and Path(p).with_name("probs_dev.json").exists()})
runs = {}
for d in dirs:
    jd, jt = json.load(open(Path(d) / "probs_dev.json")), json.load(open(Path(d) / "probs_test.json"))
    runs.setdefault(jd["run"], (jd["docs"], jt["docs"]))
names = sorted(runs)
assert names, "no runs found"
dev_ids = sorted(runs[names[0]][0]); test_ids = list(runs[names[0]][1])
y = np.array([runs[names[0]][0][i][1] for i in dev_ids])
PD = {n: np.array([runs[n][0][i][0] for i in dev_ids]) for n in names}
PT = {n: np.array([runs[n][1][i][0] for i in test_ids]) for n in names}
print(f"{'run':>28} {'F1@0.5':>8} {'F1 tuned':>9}")
for n in names:
    print(f"{n:>28} {macro_f1(y, (PD[n] >= .5).astype(int)):8.4f} {best_threshold(y, PD[n])[1]:9.4f}")
members, best = [], -1.0
for _ in range(10):
    cand = [(best_threshold(y, np.mean([PD[m] for m in members + [n]], 0))[1], n) for n in names]
    f, n = max(cand)
    if f <= best + 1e-6:
        break
    members.append(n); best = f
    print(f"+ {n:>28} -> dev F1 (tuned) {f:.4f}")
pd_ = np.mean([PD[m] for m in members], 0); pt_ = np.mean([PT[m] for m in members], 0)
thr, f = best_threshold(y, pd_)
h0, h1 = honest_threshold(y, pd_)
thr = thr if h1 >= h0 else 0.5
pr, rc = prec_rec(y, (pd_ >= thr).astype(int))
print(f"ensemble {members}: dev F1@0.5 {macro_f1(y, (pd_ >= .5).astype(int)):.4f} | tuned {f:.4f} (thr {thr:.3f}) | honest {h0:.4f}->{h1:.4f} | P {pr:.3f} R {rc:.3f}")
out = [{"id": i, "pred": int(v >= thr)} for i, v in zip(test_ids, pt_)]
json.dump(out, open(OUT / "prediction_ensemble.json", "w"))
with zipfile.ZipFile(OUT / "submission_ensemble.zip", "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(OUT / "prediction_ensemble.json", arcname="prediction.json")
json.dump(dict(members=members, dev_f1_tuned=f, thr=thr, honest=[h0, h1], test_pos_rate=float(np.mean([o["pred"] for o in out]))),
          open(OUT / "ensemble_results.json", "w"), indent=1)
print("wrote", OUT / "submission_ensemble.zip", "| test 1-rate", np.mean([o["pred"] for o in out]))

Writing cj_ens.py


In [6]:
# Fix 3: enc-dec padding mask must be re-applied when gradient checkpointing recomputes a block
s = open("cj_lib.py").read()
old_ck = "                h = checkpoint(blk, h, cos, sin, use_reentrant=False) if ckpt else blk(h, cos, sin)\n"
assert old_ck in s
s = s.replace(old_ck, "                h = checkpoint(_call_block, blk, h, cos, sin, valid if enc else None, use_reentrant=False) if ckpt else blk(h, cos, sin)\n")
s = s.replace("def run_backbone(bb, x, valid, train_from=0, ckpt=True):\n",
              "def _call_block(blk, h, cos, sin, mask):\n"
              "    if mask is not None:\n"
              "        _ENC_MASK[0] = mask\n"
              "    return blk(h, cos, sin)\n\n\n"
              "def run_backbone(bb, x, valid, train_from=0, ckpt=True):\n", 1)
open("cj_lib.py", "w").write(s)
print("patched enc-dec checkpointing")
# Fix 4: no gradient checkpointing for the 6-layer enc-dec encoder (avoids the recomputation/padding-mask clash)
s = open("cj_lib.py").read()
old4 = 'bool(hcfg.get("ckpt", True))'
assert s.count(old4) == 1, "fix 4 target not found"
s = s.replace(old4, old4 + ' and bb.kind != "encdec"')
open("cj_lib.py", "w").write(s)
print("patched: enc-dec runs without checkpointing")

patched enc-dec checkpointing
patched: enc-dec runs without checkpointing


In [7]:
# ===== Phase 4 driver: every experiment runs as its own process, one per T4, then KD (needs the teachers), then the ensemble =====
import os, sys, subprocess, time, glob, shutil, json, py_compile
from pathlib import Path
import torch
for f in ("cj_lib.py", "cj_peft.py", "cj_run.py", "cj_ens.py"):
    py_compile.compile(f, doraise=True)
SMOKE = False        # True = tiny 1-epoch pipeline check (~10 min). Set False for the real runs (~5-6 h on 2 x T4).
os.environ["HF_TOKEN"] = HF_TOKEN or ""
NGPU = max(1, torch.cuda.device_count())
QUEUES = {0: [("full", "gqa"), ("lorma", "gqa"), ("bitfit", "mla")],
          1: [("partial", "retnet"), ("adapter", "encdec"), ("adalora", "retnet")]}
if NGPU == 1:
    QUEUES = {0: QUEUES[0] + QUEUES[1]}
FINAL = {0: [("kd", "gqa")]}
LOG = Path("/kaggle/working/logs"); LOG.mkdir(parents=True, exist_ok=True)
T0 = time.time()

def tail(path, n=1):
    try:
        return open(path).read().strip().splitlines()[-n:]
    except Exception:
        return []

def run_queues(queues):
    pend = {g: list(q) for g, q in queues.items()}
    cur, status, last = {g: None for g in queues}, {}, 0
    while True:
        for g in queues:
            if cur[g] is not None and cur[g][0].poll() is not None:
                p, f, name = cur[g]; f.close(); status[name] = p.returncode
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} finished {name} rc={p.returncode}\n    " + "\n    ".join(tail(LOG / f"{name}.log", 4)), flush=True)
                cur[g] = None
            if cur[g] is None and pend[g]:
                m, a = pend[g].pop(0); name = f"{m}_{a}"
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), CJ_SMOKE="1" if SMOKE else "0", PYTHONUNBUFFERED="1")
                f = open(LOG / f"{name}.log", "w")
                cur[g] = (subprocess.Popen([sys.executable, "cj_run.py", m, a], stdout=f, stderr=subprocess.STDOUT, env=env), f, name)
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} started {name}", flush=True)
        if all(c is None for c in cur.values()):
            return status
        if time.time() - last > 300:
            last = time.time()
            for g, c in cur.items():
                if c is not None:
                    print(f"[{(time.time() - T0) / 60:6.1f} min]   GPU{g} {c[2]}: {tail(LOG / (c[2] + '.log'))}", flush=True)
        time.sleep(15)

print("GPUs:", NGPU, "| queues:", QUEUES, "| smoke:", SMOKE, flush=True)
status = run_queues(QUEUES)
status.update(run_queues(FINAL))
print("\nreturn codes:", status)
if not SMOKE:
    subprocess.run([sys.executable, "cj_ens.py"])
sub = Path("/kaggle/working/submissions"); sub.mkdir(exist_ok=True)
for z in glob.glob("/kaggle/working/cj_*/submission.zip") + glob.glob("/kaggle/working/submission_ensemble.zip"):
    p = Path(z); shutil.copy(z, sub / (f"{p.parent.name}.zip" if p.parent.name.startswith("cj_") else p.name))
print(sorted(os.listdir(sub)))
print("\n=== SUMMARIES ===")
for lg in sorted(LOG.glob("*.log")):
    for line in open(lg).read().splitlines():
        if line.startswith(("SUMMARY", "KD:", "DEV ", "overlap", "train:", "dev:", "test:", "class weights")) or "Traceback" in line or "Error" in line:
            print(lg.stem, "|", line)
print("ALL_DONE")

GPUs: 2 | queues: {0: [('full', 'gqa'), ('lorma', 'gqa'), ('bitfit', 'mla')], 1: [('partial', 'retnet'), ('adapter', 'encdec'), ('adalora', 'retnet')]} | smoke: False
[   0.0 min] GPU0 started full_gqa
[   0.0 min] GPU1 started partial_retnet
[   0.0 min]   GPU0 full_gqa: []
[   0.0 min]   GPU1 partial_retnet: []
[   5.0 min]   GPU0 full_gqa: ['train after id de-duplication: 32305 (removed 5082)']
[   5.0 min]   GPU1 partial_retnet: ['train after id de-duplication: 32305 (removed 5082)']
[  10.0 min]   GPU0 full_gqa: ['  ep 1 step 125/1875 loss 0.7426 | 10,388 tok/s | 3.9 min']
[  10.0 min]   GPU1 partial_retnet: ['  ep 1 step 150/1875 loss 0.7452 | 12,012 tok/s | 4.1 min']
[  15.0 min]   GPU0 full_gqa: ['  ep 1 step 275/1875 loss 0.7238 | 10,161 tok/s | 9.0 min']
[  15.0 min]   GPU1 partial_retnet: ['  ep 1 step 300/1875 loss 0.7228 | 11,958 tok/s | 8.4 min']
[  20.0 min]   GPU0 full_gqa: ['  ep 1 step 350/1875 loss 0.7097 | 8,196 tok/s | 14.3 min']
[  20.0 min]   GPU1 partial_retnet:

Traceback (most recent call last):
  File "/kaggle/working/cj_ens.py", line 19, in <module>
    PD = {n: np.array([runs[n][0][i][0] for i in dev_ids]) for n in names}
                       ~~~~~~~~~~^^^
KeyError: '1950_14'


['cj_adalora_retnet.zip', 'cj_adapter_encdec.zip', 'cj_bitfit_mla.zip', 'cj_full_gqa.zip', 'cj_kd_gqa.zip', 'cj_lorma_gqa.zip', 'cj_partial_retnet.zip']

=== SUMMARIES ===
adalora_retnet | train: 37387 docs | raw list | keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'] | example id='2020_1' label=0 text=' Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by t'
adalora_retnet | dev: 3505 docs | raw list | keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'] | example id='1989_75' label=0 text='civil appellate jurisdiction civil appeal number 1709\nof\n1988.\nfrom the judgment and order dated 9.1'
adalora_retnet | test: 1517 docs | raw list | keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'] | example id='1986_397' label=1 text='civil appellate jurisdiction civil appeal number 1257\n nl  of 1985.\nfrom the judgment and order date'
